# 02b. 가격을 뺀 동 유형화(02 추가안)

**질문(02 보조):** 가격을 쓰지 않고 동의 입지·교육·주거 특성만으로 묶어도 가격 수준이 다른 유형이 나타나는가?

**방법과 지표**
- 입력(가격 제외, `dong_features.parquet`): 업무지구 최단 통근시간(`commute_min`), log(1+입시·보습 학원 수), log(최근접 지하철역 거리), 반경 500m 버스정류소 수, 거래 단지의 연식(2026 − 건축년도 중앙값), log(단지 세대수 중앙값).
- 표준화 후 k-means(k=2~6), 관성(엘보우)·실루엣·군집 크기 비교. **주 k는 실루엣 최댓값(사전 규칙).** 안정성은 동 부트스트랩 200회 ARI.
- 유형을 정한 **뒤에** 가격을 붙인다: 동 보정 가격 수준(2024-01~2026-09, 동 FE + 월 FE + 면적·연식·층 구간), 유형별 월 지수(2019-01~2026-09, 단지 FE)의 변동성(월 변화율 표준편차)과 고점→저점 하락률.
- 유형과 02의 가격 급지(2018년 기준, 구 단위)가 얼마나 겹치는지 교차표와 ARI로 본다.
- 지도는 타일 없이 동 대표 좌표 점으로 그린다.

**사용한 교재 장:** 7장 k-means, 8장 실루엣·엘보우·지도 시각화.
**교재에서 바꾼 점과 이유:** 군집 안정성(부트스트랩 ARI)을 추가했다. 군집 결과가 표본에 따라 흔들리는지 확인해야 유형 이름을 붙일 수 있기 때문이다. 가격은 군집 입력에서 빼고 사후 비교에만 쓴다(조율안 02 추가안).

**주의:** 입력 특징은 모두 현재 시점 자료다(`dong_features_meta.csv`). 유형별 가격 차이는 통계적 연관이며, 특정 특성이 가격을 만든다는 뜻이 아니다.

In [1]:
import os
import sys
from pathlib import Path


def _find_root():
    env = os.environ.get("REANALYSIS_ROOT")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "AGENTS.md").exists():
            return p
    raise FileNotFoundError("AGENTS.md가 있는 프로젝트 루트를 찾지 못했습니다")


ROOT = _find_root()
sys.path.insert(0, str(ROOT / "src" / "재검증"))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import reanalysis_utils as ru

SEED = ru.SEED
DATA = ROOT / "data"
plt = ru.set_korean_font()

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from scipy import stats

OUT = ru.out_dir(ROOT, "02b")
feat = pd.read_parquet(ROOT / "outputs" / "재검증" / "00_정제" / "dong_features.parquet")
tx = ru.load_tx(ROOT)
nob = tx[tx["in_base_nobulk"]].reset_index(drop=True)
feat["x_commute"] = feat["commute_min"]
feat["x_acad"] = np.log1p(feat["acad_exam"])
feat["x_subway"] = np.log(feat["subway_dist_m"])
feat["x_bus"] = feat["bus_n_500m"]
feat["x_age"] = 2026 - feat["median_build_year"]
feat["x_hh"] = np.log(feat["apt_households_median"].clip(lower=1))
XCOLS = ["x_commute", "x_acad", "x_subway", "x_bus", "x_age", "x_hh"]
XLAB = {"x_commute": "최단 통근(분)", "x_acad": "log(1+입시학원)", "x_subway": "log 역거리(m)", "x_bus": "버스정류소(500m)",
        "x_age": "연식(년)", "x_hh": "log 세대수 중앙값"}
d = feat.dropna(subset=XCOLS).reset_index(drop=True)
pd.Series({"dongs_total": len(feat), "dongs_complete": len(d),
           "dropped_commute_missing": int(feat["x_commute"].isna().sum()),
           "dropped_other": int(len(feat) - len(d) - feat["x_commute"].isna().sum())}).to_frame("n")

,n
dongs_total,338
dongs_complete,272
dropped_commute_missing,62
dropped_other,4


통근시간이 없는 동(00에서 62개)과 단지정보·좌표가 없는 동은 군집에서 빠진다. 빠진 동은 종로·중·성북·용산 등 도심 구에 몰려 있어, 유형화 결과는 **통근시간이 있는 동에 대한 결과**다.

In [3]:
Z = StandardScaler().fit_transform(d[XCOLS])
krows, labs = [], {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=50, random_state=SEED).fit(Z)
    labs[k] = km.labels_
    sz = np.bincount(km.labels_)
    krows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(Z, km.labels_),
                  "min_size": int(sz.min()), "sizes": "-".join(map(str, sorted(sz, reverse=True)))})
kt = pd.DataFrame(krows)
K = int(kt.loc[kt["silhouette"].idxmax(), "k"])


def boot_ari(k, B=200):
    rng = np.random.default_rng(SEED)
    out = []
    for _ in range(B):
        idx = rng.choice(len(Z), size=len(Z), replace=True)
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z[idx])
        out.append(adjusted_rand_score(labs[k], km.predict(Z)))
    return np.array(out)


ari = {k: boot_ari(k) for k in range(2, 7)}
kt["boot_ari_mean"] = kt["k"].map({k: v.mean() for k, v in ari.items()})
kt["boot_ari_p05"] = kt["k"].map({k: np.percentile(v, 5) for k, v in ari.items()})
kt

,k,inertia,silhouette,min_size,sizes,boot_ari_mean,boot_ari_p05
0,2,1248.477351,0.221381,135,137-135,0.890097,0.777088
1,3,1096.364662,0.200632,71,122-79-71,0.704972,0.420418
2,4,985.072830,0.191029,33,104-72-63-33,0.544961,0.359451
3,5,895.056034,0.179566,30,92-58-48-44-30,0.577054,0.379560
4,6,815.703111,0.196944,10,89-58-46-40-29-10,0.609180,0.416663


실루엣이 가장 높은 k를 주 k로 쓴다(위 표, `K`). `boot_ari_mean`은 동을 재표본해 다시 군집했을 때 원래 분류와의 일치도다.

In [4]:
d["type_raw"] = labs[K]
order = d.groupby("type_raw")["x_commute"].mean().sort_values().index
tmap = {o: i + 1 for i, o in enumerate(order)}
d["type"] = d["type_raw"].map(tmap)
prof = d.groupby("type")[XCOLS].mean().rename(columns=XLAB)
prof["dongs"] = d.groupby("type").size()
prof["n_gu"] = d.groupby("type")["gu"].nunique()
prof["acad_exam_median"] = d.groupby("type")["acad_exam"].median()
prof["subway_dist_median_m"] = d.groupby("type")["subway_dist_m"].median()
prof.round(2)

,최단 통근(분),log(1+입시학원),log 역거리(m),버스정류소(500m),연식(년),log 세대수 중앙값,dongs,n_gu,acad_exam_median,subway_dist_median_m
type,,,,,,,,,,
1,19.84,1.54,5.87,26.03,20.67,5.87,137,16,4.0,369.57
2,36.40,3.95,6.33,19.44,24.91,6.01,135,25,59.0,520.67


유형 번호는 평균 최단 통근시간이 짧은 순서다(1=가장 짧음). 위 표는 유형별 입력 특징 평균(원래 단위)이다.

## 2. 유형을 정한 뒤 가격 비교

In [5]:
REC = ru.ym_range("2024-01", "2026-09")
lev, lev_res = ru.dong_level(nob[nob["contract_ym"].isin(REC)].reset_index(drop=True))
d = d.merge(lev, on="dong_id", how="left")
MIN_TX = 30
d["price_ok"] = d["n_tx"].fillna(0) >= MIN_TX
dp = d[d["price_ok"]]
grp = dp.groupby("type")["adj_log_level"]
price_tab = pd.DataFrame({"dongs": grp.size(), "mean_level_pct": (np.exp(grp.mean()) - 1) * 100,
                          "median_level_pct": (np.exp(grp.median()) - 1) * 100,
                          "p25_pct": (np.exp(grp.quantile(.25)) - 1) * 100, "p75_pct": (np.exp(grp.quantile(.75)) - 1) * 100})
kw = stats.kruskal(*[g.to_numpy() for _, g in grp])
ss_between = sum(len(g) * (g.mean() - dp["adj_log_level"].mean()) ** 2 for _, g in grp)
eta2 = ss_between / ((dp["adj_log_level"] - dp["adj_log_level"].mean()) ** 2).sum()
gu_g = dp.groupby("gu")["adj_log_level"]
eta2_gu = sum(len(g) * (g.mean() - dp["adj_log_level"].mean()) ** 2 for _, g in gu_g) / ((dp["adj_log_level"] - dp["adj_log_level"].mean()) ** 2).sum()
display(price_tab.round(1))
pd.Series({"kruskal_H": kw.statistic, "kruskal_p": kw.pvalue, "eta2_type": eta2, "eta2_gu_reference": eta2_gu,
           "dongs_with_price": len(dp), "tx_used": int(lev_res["n"])}).round(4)

,dongs,mean_level_pct,median_level_pct,p25_pct,p75_pct
type,,,,,
1,122,10.8,11.5,-11.8,38.3
2,133,-4.0,-12.9,-30.2,23.9


kruskal_H                16.7233
kruskal_p                 0.0000
eta2_type                 0.0359
eta2_gu_reference         0.6766
dongs_with_price        255.0000
tx_used              181874.0000
dtype: float64

`mean_level_pct`는 같은 면적·연식·층 구성으로 맞춘 동 가격 수준을 서울 평균 대비 %로 바꾼 값이다(2024-01~2026-09, 거래 30건 이상 동). `eta2_type`은 동 가격 수준의 분산 중 유형으로 갈리는 몫, `eta2_gu_reference`는 비교용으로 구(25개)로 갈리는 몫이다. 유형 수가 구 수보다 훨씬 적다는 점을 감안해 읽는다.

In [6]:
AN = ru.ym_range("2019-01", "2026-09")
tmp = nob[nob["contract_ym"].isin(AN)].merge(d[["dong_id", "type"]], on="dong_id", how="inner")
tid = []
for t, g in tmp.groupby("type"):
    gi, _ = ru.time_dummy_index(g.reset_index(drop=True), AN, fe="complex_id", char_refs=ru.CHAR_REFS_NOAGE, with_se=False)
    gi["type"] = int(t)
    tid.append(gi)
tid = pd.concat(tid, ignore_index=True)
tid["ret"] = tid.groupby("type")["log_index"].diff() * 100
vol = tid.groupby("type").agg(ret_sd=("ret", "std"), tx_per_month=("n", "median"))
pk = tid[tid["month"] <= "2023-12"].loc[lambda x: x.groupby("type")["index"].idxmax()].set_index("type")
vol["peak_month"] = pk["month"]
vol["drawdown_pct"] = [
    (tid[(tid["type"] == t) & (tid["month"] > pk.loc[t, "month"]) & (tid["month"] <= "2024-12")]["index"].min()
     / pk.loc[t, "index"] - 1) * 100 for t in vol.index]
vol["cum_2019_01_to_2026_09_pct"] = tid[tid["month"] == AN[-1]].set_index("type")["index"] - 100
vol.round(2)

,ret_sd,tx_per_month,peak_month,drawdown_pct,cum_2019_01_to_2026_09_pct
type,,,,,
1,1.99,714.0,2021-10,-21.36,76.04
2,1.91,3127.0,2021-09,-25.52,73.52


`ret_sd`는 유형별 월 지수 변화율(log×100)의 표준편차, `drawdown_pct`는 2023년까지의 고점에서 2024년까지의 저점까지 하락률이다. 유형마다 거래 수가 달라 월 지수의 측정 잡음도 다르므로, `ret_sd` 차이는 거래 수(`tx_per_month`)와 함께 읽는다.

## 3. 가격 급지(02)와의 겹침

In [7]:
g02 = pd.read_csv(ROOT / "outputs" / "재검증" / "02" / "gu_groups.csv")
tier_cols = [c for c in g02.columns if c.startswith("tier_k")]
d = d.merge(g02[["gu"] + tier_cols], on="gu", how="left")
over = []
for c in tier_cols:
    over.append({"tier_col": c, "ari_type_vs_tier": adjusted_rand_score(d[c], d["type"]),
                 "cramers_v": np.sqrt(stats.chi2_contingency(pd.crosstab(d["type"], d[c]))[0] /
                                      (len(d) * (min(d["type"].nunique(), d[c].nunique()) - 1)))})
over = pd.DataFrame(over)
display(over.round(3))
pd.crosstab(d["type"], d[tier_cols[-1]], margins=True)

,tier_col,ari_type_vs_tier,cramers_v
0,tier_k2,0.063,0.252
1,tier_k4,0.090,0.428


tier_k4,1,2,3,4,All
type,,,,,
1,4,52,64,17,137
2,18,26,33,58,135
All,22,78,97,75,272


ARI가 0에 가까우면 비가격 유형과 가격 급지가 거의 겹치지 않는다는 뜻이고, Cramér's V는 0~1 범위의 연관 크기다. 교차표 행은 비가격 유형, 열은 02의 가격 급지(구 단위, 1=2018년 가격 수준 최상)다.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
cmap = plt.get_cmap("tab10")
for t, g in d.groupby("type"):
    axes[0].scatter(g["lon"], g["lat"], s=18, color=cmap(t - 1), label=f"유형{t} ({len(g)}개 동)")
axes[0].set_xlabel("경도")
axes[0].set_ylabel("위도")
axes[0].legend(fontsize=8)
axes[0].set_title(f"비가격 동 유형 (k={K}, 동 대표 좌표)", fontsize=10)
bp = [dp.loc[dp["type"] == t, "adj_log_level"].to_numpy() * 100 for t in sorted(dp["type"].unique())]
axes[1].boxplot(bp, labels=[f"유형{t}" for t in sorted(dp["type"].unique())])
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set_ylabel("동 보정 가격 수준(서울 평균 대비 log×100)")
axes[1].set_title("유형별 동 보정 가격 수준 (2024-01~2026-09)", fontsize=10)
fig.text(0.01, -0.02, f"동 {len(d)}개(가격 비교는 거래 30건 이상 {len(dp)}개). 자료: outputs/재검증/02b/fig_type_map.csv, fig_type_price_box.csv", fontsize=8)
fig.tight_layout()
ru.save_png(fig, OUT, "fig_type_map_price.png")
plt.close(fig)

## 4. 저장

In [9]:
mapdf = d[["dong", "gu", "dong_id", "lat", "lon", "type", "adj_log_level", "n_tx", "price_ok"] + XCOLS]
ru.save_chart_csv(mapdf, OUT, "fig_type_map.csv")
box = dp.groupby("type")["adj_log_level"].describe(percentiles=[.05, .25, .5, .75, .95]).reset_index()
ru.save_chart_csv(box, OUT, "fig_type_price_box.csv")
ru.save_chart_csv(kt, OUT, "fig_k_selection.csv")
ru.save_chart_csv(prof.reset_index(), OUT, "table_type_profile.csv")
ru.save_chart_csv(tid[["type", "month", "index", "log_index", "ret", "n"]], OUT, "fig_type_index.csv")
ru.save_chart_csv(vol.reset_index(), OUT, "table_type_volatility.csv")
ru.save_chart_csv(over, OUT, "table_type_vs_tier.csv")

rows = []


def add(key, label, value, unit, n, ps, pe, note, src, claim="통계적 연관"):
    rows.append({"key": key, "label": label, "value": float(value), "unit": unit, "sample_n": int(n),
                 "period": f"{ps}~{pe}", "period_start": ps, "period_end": pe, "split_type": "없음",
                 "claim_level": claim, "source_file": src, "note": note})


ktk = kt.set_index("k")
add("k_main", "실루엣 기준 주 k", K, "개", len(d), "2026-10", "2026-11", "입력은 현재 시점 동 특징(기간은 자료 기준일)", "fig_k_selection.csv", "단순 변화")
add("silhouette_k_main", f"k={K} 실루엣", ktk.loc[K, "silhouette"], "점수", len(d), "2026-10", "2026-11", "", "fig_k_selection.csv", "단순 변화")
add("boot_ari_k_main", f"k={K} 동 부트스트랩 ARI 평균", ktk.loc[K, "boot_ari_mean"], "ARI", len(d), "2026-10", "2026-11",
    f"200회, 5% 분위 {ktk.loc[K, 'boot_ari_p05']:.2f}", "fig_k_selection.csv", "단순 변화")
add("eta2_type_price", "동 보정 가격 수준 분산 중 유형 몫(η²)", eta2, "비율", len(dp), "2024-01", "2026-09",
    f"Kruskal-Wallis p={kw.pvalue:.2g}, 구 몫(참고) {eta2_gu:.3f}", "fig_type_price_box.csv")
for t, r in price_tab.iterrows():
    add(f"type{t}_mean_level_pct", f"유형{t} 동 보정 가격 수준(서울 평균 대비)", r["mean_level_pct"], "%", int(r["dongs"]), "2024-01", "2026-09",
        f"동 {int(r['dongs'])}개, IQR {r['p25_pct']:.1f}~{r['p75_pct']:.1f}%", "fig_type_price_box.csv")
for t, r in vol.iterrows():
    add(f"type{t}_ret_sd", f"유형{t} 월 지수 변화율 표준편차", r["ret_sd"], "log포인트(×100)", int(r["tx_per_month"]), "2019-01", "2026-09",
        f"월 거래 중앙값 {r['tx_per_month']:.0f}건, 고점→저점 {r['drawdown_pct']:.1f}%", "table_type_volatility.csv", "단순 변화")
for _, r in over.iterrows():
    add(f"ari_type_vs_{r['tier_col']}", f"비가격 유형과 02 급지({r['tier_col']})의 ARI", r["ari_type_vs_tier"], "ARI", len(d), "2018-01", "2026-11",
        f"Cramér's V {r['cramers_v']:.3f}", "table_type_vs_tier.csv")
summary = ru.save_summary(rows, OUT)
ru.save_meta({
    "question": "가격을 빼고 동 특성만으로 묶어도 가격 수준이 다른 유형이 나타나는가?(02 추가안)",
    "method": "dong_features의 통근·학원·지하철·버스·연식·세대수 표준화 → k-means(k=2~6, 실루엣·엘보우, 동 부트스트랩 ARI) → 유형별 동 보정 가격 수준(2024-01~2026-09), 유형별 단지FE 지수 변동성(2019-01~2026-09), 02 급지와 ARI·Cramér's V",
    "textbook_chapters": ["7장 k-means", "8장 실루엣·엘보우·지도 시각화"],
    "sample_period": "특징: 현재 시점(2026-10~11 기준), 가격: 2024-01~2026-09, 지수: 2019-01~2026-09",
    "sample_size": {"dongs_clustered": len(d), "dongs_with_price": len(dp)},
    "data_files": ["outputs/재검증/00_정제/dong_features.parquet", "outputs/재검증/00_정제/transactions.parquet", "outputs/재검증/02/gu_groups.csv"],
    "limitations": ["통근시간 결측 동(도심 62개 등)은 군집에서 빠졌다", "입력 특징은 모두 현재 시점 자료", "유형별 가격 차이는 연관이며 원인이 아니다",
                    "유형 이름은 입력 특징 평균으로 붙인 서술용", "2026-08~09는 잠정 월"],
}, OUT)
summary[["key", "value", "note"]]

,key,value,note
0,k_main,2.000000,입력은 현재 시점 동 특징(기간은 자료 기준일)
1,silhouette_k_main,0.221381,
2,boot_ari_k_main,0.890097,"200회, 5% 분위 0.78"
3,eta2_type_price,0.035870,"Kruskal-Wallis p=4.3e-05, 구 몫(참고) 0.677"
4,type1_mean_level_pct,10.765581,"동 122개, IQR -11.8~38.3%"
5,type2_mean_level_pct,-4.004203,"동 133개, IQR -30.2~23.9%"
6,type1_ret_sd,1.987015,"월 거래 중앙값 714건, 고점→저점 -21.4%"
7,type2_ret_sd,1.905323,"월 거래 중앙값 3127건, 고점→저점 -25.5%"
8,ari_type_vs_tier_k2,0.062862,Cramér's V 0.252
9,ari_type_vs_tier_k4,0.090206,Cramér's V 0.428


## 결론
- **군집 구조는 약하다(단순 변화).** 통근시간이 있는 272개 동을 k=2~6으로 나눈 실루엣은 0.18~0.22로 낮았고, 사전 규칙(실루엣 최댓값)에 따라 k=2를 골랐다(실루엣 0.221, 동 부트스트랩 ARI 평균 0.89). 동 특성이 뚜렷한 덩어리로 나뉘기보다 연속적으로 퍼져 있다는 뜻이다(`fig_k_selection.csv`).
- **두 유형(단순 변화):** 유형1(137개 동)은 업무지구 최단 통근 평균 19.8분, 입시·보습 학원 중앙값 4개, 역까지 중앙값 370m, 평균 연식 20.7년이다. 유형2(135개 동)는 통근 36.4분, 학원 중앙값 59개, 역까지 521m, 연식 24.9년, 세대수가 조금 더 크다(`table_type_profile.csv`). 즉 "도심 가까운 소규모 학원 동"과 "외곽 학원 밀집·노후 대단지 동"으로 갈렸다.
- **가격과의 연관(통계적 연관):** 2024-01~2026-09 동 보정 가격 수준은 유형1이 서울 평균 대비 +10.8%, 유형2가 −4.0%였다(거래 30건 이상 255개 동, Kruskal-Wallis p<0.001). 그러나 동 가격 분산 중 유형으로 갈리는 몫(η²)은 0.036으로, 구(25개)로 갈리는 몫 0.68보다 훨씬 작고 유형 안 편차(IQR 약 50%p)가 크다(`fig_type_price_box.csv`).
- **급지와의 겹침:** 비가격 유형과 02의 가격 급지 사이 ARI는 0.06(k=2)~0.09(k=4)로 낮았고 Cramér's V는 0.25~0.43이었다(`table_type_vs_tier.csv`). **가격을 빼고 묶으면 가격 급지는 거의 다시 나타나지 않는다.**
- 변동성: 두 유형의 월 지수 변화율 표준편차는 1.99와 1.91로 비슷했고, 2021년 고점→저점 하락률은 −21.4%와 −25.5%였다(`table_type_volatility.csv`).

## 한계
- 통근시간 결측 동 62개(종로·성북·중·용산 등 도심)와 세대수 정보 없는 동 4개는 군집에서 빠졌다. 도심 동이 빠져 유형1의 성격이 실제보다 좁게 잡혔을 수 있다.
- 입력 특징은 모두 현재 시점 자료다. 2019년 이후 지수와 비교할 때 시점이 맞지 않는다.
- 유형 이름은 입력 특징 평균으로 붙인 서술용이며, 유형 간 가격 차이는 연관이다(역인과·누락 변수 가능).
- 실루엣이 낮아 유형 경계 근처 동의 소속은 불안정하다.